# NB10 — Single-cell context; definition and freezing of S, M and R
**Run this after NB05 and before NB09/NB11.** It fixes the gene sets that the survival analysis will use.

**Dataset:** GSE131928 (Neftel 2019; processed Smart-seq2 and 10X matrices). Only one single-cell dataset is used, so
the blueprint rule "malignant-expressed in >= 2 datasets" is relaxed to `sc.min_datasets = 1` (stated limitation).

**Steps (all fixed before any survival model):**
1. Keep adult tumours. Patient IDs come from cell names (`MGH###`, `BT###`; 10X names that start with digits such as
   `105A_...` are read as `MGH105`). Adult rule = `config.sc.adult_patient_regex` (default `^MGH`). The patient table is
   printed so you can check it.
2. **Malignant calls** with an inferCNV-style score: genes ordered by genome position (GENCODE v23), expression
   relative to reference cells (top 5% for immune/oligodendrocyte markers), 100-gene moving average **within each
   chromosome**, score = mean squared signal. Malignant = score > reference mean + `cnv_z_threshold` SD and not a
   reference cell. Check: malignant cells should show chr7 gain and chr10 loss.
3. **Patient-level detection:** for each S gene, the fraction of a patient's malignant cells with expression > 0.
   **M** = S genes detected in >= 10% of malignant cells in >= 50% of eligible patients (>= 50 malignant cells).
   **R** = S minus M (includes S genes not measured in the single-cell data).
4. Freeze S, M, R with SHA-256 checksums. Later notebooks refuse to run if these files change.

Memory: about 6–8 GB RAM. If the runtime crashes, use a high-RAM runtime, or move the 10X file out of
`02_Raw_Datasets/GSE131928/extracted/` to run on Smart-seq2 only (and record that).

In [ ]:
import os, sys, re, gc
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("NB10 needs the GSE131928 download (full mode).")
SC = CFG["sc"]; RAW = P("raw"); OUT = P("sc")
S = pd.read_csv(P("final", "S_candidate_genes.csv")).gene.astype(str).tolist()
print("Resources:", resource_report(), "| S genes:", len(S))

In [ ]:
ext = RAW / "GSE131928" / "extracted"
files = sorted(p for p in ext.rglob("*") if p.is_file()) if ext.exists() else []
require(bool(files), "GSE131928 files not found. Run NB01 in full mode.")
print("Files:", [p.name for p in files])
cand = {"Smartseq2": [p for p in files if "smartseq2" in p.name.lower() and "tpm" in p.name.lower()],
        "10X": [p for p in files if "10x" in p.name.lower() and "tpm" in p.name.lower()]}
require(any(cand.values()), "No processed TPM matrices matched '*Smartseq2*TPM*' or '*10X*TPM*'. Adjust the patterns to the file names above.")


def read_mat(p, chunksize=3000):
    """Read a genes x cells TSV as float32 in chunks (keeps memory low). Handles a header with or without a gene column."""
    probe = pd.read_csv(p, sep="\t", nrows=3, compression="infer")
    implicit_index = not isinstance(probe.index, pd.RangeIndex)
    if implicit_index:          # header has one field fewer than the data rows
        kw = dict(dtype={c: np.float32 for c in probe.columns})
    else:
        kw = dict(index_col=0, dtype={c: np.float32 for c in probe.columns[1:]})
    parts = [ch for ch in pd.read_csv(p, sep="\t", compression="infer", chunksize=chunksize, **kw)]
    df = pd.concat(parts)
    df.index = df.index.astype(str).str.strip().str.strip("'\"")
    df = df[~df.index.duplicated(keep="first")]
    vmin = float(np.nanmin(df.values))
    require(vmin >= 0, f"{p.name}: negative values (min {vmin}); the matrix is centred, so detection fractions would be invalid.")
    return df


def patient_of(cell):
    s = str(cell)
    m = re.match(r"^(MGH\d+|BT\d+)", s)
    if m:
        return m.group(1)
    m = re.match(r"^(\d{2,4})[A-Za-z]?[-_.]", s)          # 10X style, e.g. 105A_AAACC... -> MGH105
    if m:
        return "MGH" + m.group(1)
    m = re.match(r"^([A-Za-z]+\d+)", s)
    return m.group(1) if m else "unknown"


adult_re = re.compile(SC["adult_patient_regex"])
mats = {}
for k, v in cand.items():
    if not v:
        continue
    m = read_mat(v[0])
    pats = pd.Series([patient_of(c) for c in m.columns], index=m.columns)
    print(f"\n{k}: {m.shape[0]} genes x {m.shape[1]} cells; value range {float(m.values.min()):.2f}..{float(m.values.max()):.2f}")
    print("cells per patient:", pats.value_counts().to_dict())
    keep = pats.index[pats.map(lambda x: bool(adult_re.match(x)))]
    log(f"{k}: {len(keep)} of {m.shape[1]} cells pass the adult rule ({SC['adult_patient_regex']}).")
    if len(keep) < 500:
        log(f"{k}: fewer than 500 adult cells; dataset skipped. Check the patient table above.", "WARN")
        del m; gc.collect(); continue
    mats[k] = m[keep]
    del m; gc.collect()
require(bool(mats), "No dataset kept any adult cells; adjust config.sc.adult_patient_regex using the patient tables above.")

In [ ]:
# ---- gene coordinates and CNV-style malignant calls ------------------------------------------------------
pm = pd.read_csv(RAW / "TOIL" / "gencode.v23.annotation.gene.probemap", sep="\t")
c_gene = find_col(pm, ["gene"]); c_chr = find_col(pm, ["chrom"]); c_st = find_col(pm, ["chromStart"])
coord = pm[[c_gene, c_chr, c_st]].rename(columns={c_gene: "gene", c_chr: "chrom", c_st: "start"}).drop_duplicates("gene")
coord = coord[coord.chrom.astype(str).str.match(r"^chr([0-9]+)$")].copy()
coord["chr_n"] = coord.chrom.str.replace("chr", "").astype(int)
coord = coord.sort_values(["chr_n", "start"]).set_index("gene")


def cnv_scores(m, ref_frac=0.05, win=100):
    genes = [g for g in coord.index if g in m.index]
    X = m.loc[genes].to_numpy(np.float32)
    keep = (X > 0).mean(axis=1) >= 0.05
    X = X[keep]; genes = np.array(genes)[keep]
    chrs = coord.loc[genes, "chr_n"].to_numpy()
    mk = [g for g in CFG["markers"]["malignant_cnv_reference"] if g in m.index]
    require(len(mk) >= 2, f"Too few reference markers in the matrix: {mk}")
    ref_score = m.loc[mk].mean(axis=0).to_numpy()
    ref_mask = ref_score >= np.quantile(ref_score, 1 - ref_frac)
    rel = X - X[:, ref_mask].mean(axis=1, keepdims=True)
    del X
    rel -= rel.mean(axis=0, keepdims=True)
    np.clip(rel, -3, 3, out=rel)
    sm_parts, chr_parts = [], []
    for c in np.unique(chrs):
        A = rel[chrs == c].astype(np.float64)
        if A.shape[0] < 20:
            continue
        w = min(win, A.shape[0])
        cs = np.cumsum(np.vstack([np.zeros((1, A.shape[1])), A]), axis=0)
        sm_parts.append(((cs[w:] - cs[:-w]) / w).astype(np.float32)); chr_parts.append(np.full(A.shape[0] - w + 1, c))
    del rel
    sm = np.vstack(sm_parts); sm_chr = np.concatenate(chr_parts)
    score = pd.Series((sm.astype(np.float64) ** 2).mean(axis=0), index=m.columns)
    return score, ref_mask, sm, sm_chr


cell_tables, checks = [], []
for k, m in mats.items():
    pats = pd.Series([patient_of(c) for c in m.columns], index=m.columns)
    score, ref_mask, sm, sm_chr = cnv_scores(m)
    ref_cells = score.index[ref_mask]
    thr = score[ref_cells].mean() + SC["cnv_z_threshold"] * score[ref_cells].std()
    malignant = (score > thr) & ~score.index.isin(ref_cells)
    mal = malignant.to_numpy(); ref = ref_mask
    c7 = float(sm[sm_chr == 7][:, mal].mean()) if mal.any() else np.nan
    c10 = float(sm[sm_chr == 10][:, mal].mean()) if mal.any() else np.nan
    r7 = float(sm[sm_chr == 7][:, ref].mean()); r10 = float(sm[sm_chr == 10][:, ref].mean())
    print(f"{k}: reference={int(ref.sum())}, malignant={int(mal.sum())} of {len(mal)} (threshold {thr:.4f}); "
          f"chr7 signal malignant {c7:+.3f} vs reference {r7:+.3f}; chr10 malignant {c10:+.3f} vs reference {r10:+.3f}")
    checks.append({"dataset": k, "n_cells": len(mal), "n_reference": int(ref.sum()), "n_malignant": int(mal.sum()),
                   "chr7_malignant": c7, "chr7_reference": r7, "chr10_malignant": c10, "chr10_reference": r10,
                   "chr7_gain_chr10_loss_seen": bool(c7 > r7 and c10 < r10)})
    cell_tables.append(pd.DataFrame({"cell": score.index, "dataset": k, "patient": pats[score.index].values,
                                     "cnv_score": score.values, "malignant": mal, "reference": ref}))
    del sm; gc.collect()
    mats[k] = m.loc[m.index.intersection(S)]          # only S genes are needed from here on
    del m; gc.collect()
cells = pd.concat(cell_tables); cells.to_csv(OUT / "cell_calls.csv.gz", index=False)
pd.DataFrame(checks).to_csv(OUT / "cnv_sanity_check.csv", index=False)
summ = cells.groupby(["dataset", "patient"]).agg(n_cells=("cell", "size"), n_malignant=("malignant", "sum"),
                                                 n_reference=("reference", "sum")).reset_index()
summ.to_csv(OUT / "patient_cell_summary.csv", index=False)
print(summ.to_string(index=False))
if not all(c["chr7_gain_chr10_loss_seen"] for c in checks):
    log("chr7 gain / chr10 loss was not seen in every dataset; review cnv_sanity_check.csv before trusting the calls.", "WARN")

In [ ]:
# ---- patient-level detection in malignant cells; M and R ------------------------------------------------
det_rows = {}
for k, m in mats.items():
    tab = cells[cells.dataset == k].set_index("cell")
    for pat, g in tab[tab.malignant].groupby("patient"):
        if len(g) < SC["min_malignant_cells_per_patient"]:
            continue
        det_rows[(k, pat)] = (m[g.index] > 0).mean(axis=1)
require(len(det_rows) >= 3, f"Only {len(det_rows)} patients have >= {SC['min_malignant_cells_per_patient']} malignant cells; check the calls above.")
det = pd.DataFrame(det_rows); det.columns = [f"{a}|{b}" for a, b in det.columns]
det.to_csv(OUT / "S_gene_detection_by_patient.csv")      # reused by NB13
print(f"Eligible patient units: {det.shape[1]}; S genes measured in scRNA: {det.shape[0]} of {len(S)}")


def define_M(det, frac, pat_frac):
    ok = (det >= frac).mean(axis=1) >= pat_frac
    return sorted(ok[ok].index)


M = define_M(det, SC["detect_fraction_in_patient"], SC["min_patient_fraction"])
R = sorted(set(S) - set(M))
print(f"S={len(S)} | measured={det.shape[0]} | M={len(M)} | R={len(R)}")
require(len(M) >= 10 and len(R) >= 10, "M or R has fewer than 10 genes; the score analysis would be unstable.")

In [ ]:
# ---- descriptive context: detection in non-malignant and reference cells ---------------------------------
ctx_parts = []
for k, m in mats.items():
    tab = cells[cells.dataset == k].set_index("cell")
    nm = tab.index[(~tab.malignant) & (~tab.reference)]; ref = tab.index[tab.reference]
    ctx_parts.append(pd.DataFrame({f"{k}_detect_frac_nonmalignant": (m[nm] > 0).mean(axis=1) if len(nm) else np.nan,
                                   f"{k}_detect_frac_reference": (m[ref] > 0).mean(axis=1)}))
ctx = pd.concat(ctx_parts, axis=1)
ctx["detect_frac_malignant_median_patient"] = det.median(axis=1)
ctx["in_M"] = ctx.index.isin(M)
ctx.to_csv(OUT / "S_gene_cellular_context.csv")
fig, ax = plt.subplots(figsize=(5.6, 4.4))
ax.hist(det.median(axis=1), bins=40, color="#1f3a93")
ax.axvline(SC["detect_fraction_in_patient"], c="#c0392b", ls="--", label="detection threshold")
ax.set_xlabel("median per-patient detection fraction in malignant cells"); ax.set_ylabel("S genes"); ax.legend(frameon=False)
savefig(fig, "SC_S_detection_in_malignant_cells", folder="sc"); plt.close(fig)

In [ ]:
# ---- freeze ---------------------------------------------------------------------------------------------
m_path, r_path, s_path = P("final", "M_genes.csv"), P("final", "R_genes.csv"), P("final", "S_candidate_genes.csv")
new_M = pd.DataFrame({"gene": M, "set": "M"}); new_R = pd.DataFrame({"gene": R, "set": "R"})
frozen_path = P("config", "frozen_definitions.json")
if frozen_path.exists():
    same = (m_path.exists() and r_path.exists() and pd.read_csv(m_path).gene.tolist() == M and pd.read_csv(r_path).gene.tolist() == R)
    require(same, "Definitions are already frozen and this run gives a different M/R. Files were NOT overwritten. "
                  "If the change is deliberate, delete frozen_definitions.json and record the re-registration in the manuscript.")
    print("Already frozen; this run reproduces the frozen M and R:", verify_frozen()["frozen_at_utc"])
else:
    new_M.to_csv(m_path, index=False); new_R.to_csv(r_path, index=False)
    payload = freeze_definitions([s_path, m_path, r_path],
                                 {"detect_fraction_in_patient": SC["detect_fraction_in_patient"], "min_patient_fraction": SC["min_patient_fraction"],
                                  "min_malignant_cells_per_patient": SC["min_malignant_cells_per_patient"], "cnv_z_threshold": SC["cnv_z_threshold"],
                                  "adult_patient_regex": SC["adult_patient_regex"], "n_S": len(S), "n_M": len(M), "n_R": len(R),
                                  "datasets_used": list(mats), "eligible_units": int(det.shape[1])})
    print("FROZEN at", payload["frozen_at_utc"], "| sha256", payload["sha256"][:16], "...")
manifest_update("single_cell", "summary", {"n_S": len(S), "n_M": len(M), "n_R": len(R), "eligible_units": int(det.shape[1]),
                                           "datasets": list(mats), "cnv_checks": checks,
                                           "limitation": "one single-cell dataset; CNV approximation"})
print("\nNB10 complete. Next: NB09 (bulk scoring), then NB11.")